In [0]:
%pip install gspread google-auth
dbutils.library.restartPython()

In [0]:
import gspread
from google.oauth2.service_account import Credentials
import json

KEY_FILE_PATH = "/Workspace/Users/pakhei_tsang@next.co.uk/advance-mantis-398714-2168c9162641.json"

with open(KEY_FILE_PATH, "r") as f:
    creds_dict = json.load(f)

creds = Credentials.from_service_account_info(
    creds_dict,
    scopes=["https://www.googleapis.com/auth/spreadsheets"]
)
gc = gspread.authorize(creds)

SHEET_ID = "1uiv27J0YPDWqSVuN-QLpS-fOcfxAjewK5ajTd3aA8vI"
sh = gc.open_by_key(SHEET_ID)

print("Connected to:", sh.title)

In [0]:
df = (spark.read
      .format("delta")
      .load("abfss://landing@whsanalyticsdlsprodeuw.dfs.core.windows.net/streaming/landing_bonushub_event_parsed/delta/"))

df.createOrReplaceTempView("landing_bonus_hub_event_parsed")

print("Row count:", df.count())
df.printSchema()

In [0]:
reports = [
    {"tab": "Data", "name": "D.Analysis - OSR PiE",
     "area": "('Pick','Pie Station')", "event": "('COUN','MSKU','PSKU','RSIN')", "extra": ""},

    {"tab": "Data", "name": "D.Analysis - OSR Topup",
     "area": "('Topup','TopUp','PiOrQi')", "event": "('QSIN','TARS','TPUT')", "extra": ""},

    {"tab": "Data", "name": "D.Analysis - E3 Packing",
     "area": "('E3 - Packing')", "event": "('PackingParcelCompleteEvent','PackingItemScannedEvent')", "extra": ""},

    {"tab": "Data", "name": "D.Analysis - Parcel Induct",
     "area": "('Parcel Induct')", "event": "('APAR','PIND','SPAR','UPAR')", "extra": ""},

    {"tab": "Data", "name": "D.Analysis - Parcel Sortation",
     "area": None, "event": "('ParcelSortedToSack','SackMappedToPosition','SackUnMappedFromPosition')", "extra": ""},

    {"tab": "Data", "name": "D.Analysis - Inbound Decanting",
     "area": "('Inbound Decanting')", "event": "('DECN','TOPR')", "extra": ""},

    {"tab": "Data", "name": "D.Analysis - OSR Decanting",
     "area": "('OSR Decanting')", "event": "('ODEC','OTOP')", "extra": ""},

    {"tab": "Data", "name": "D.Analysis - BCR Inducting",
     "area": "('Induct from E1/E2')", "event": "('SPOS')",
     "extra": "AND EXISTS(PAYLOAD_ATTRIBUTES, x -> x LIKE '%RET%')"},

    {"tab": "Data", "name": "D.Analysis - E1/E2 Inducting",
     "area": "('Induct from E1/E2')", "event": "('SPOS')",
     "extra": "AND EXISTS(PAYLOAD_ATTRIBUTES, x -> x LIKE '%PIE4EDW%')"},
]

COLUMNS = ['Date', 'Hour', 'PAYLOAD_BONUSCODE', 'PAYLOAD_EVENTTYPE',
           'Total_Quantity', 'Total_StandardHours', 'Total_SMV',
           'Week', 'Date Time Range', 'Report Name']

In [0]:
# --- Std Hours divisor: read from Front!C2 of the linked spreadsheet (the same
#     value the Apps Script side used before processing moved into Databricks).
#     Accepts a percentage cell ("96.40%") or a bare number. `sh` is the
#     spreadsheet handle from the connection cell above. ---
_div_raw = (sh.worksheet("Front").acell("C2").value or "").strip()
if not _div_raw:
    raise ValueError("Front!C2 is empty - cannot determine the Std Hours divisor.")
divisor = float(_div_raw.rstrip('%')) / 100 if _div_raw.endswith('%') else float(_div_raw)
if divisor == 0:
    divisor = 1.0
print(f"Divisor (from Front!C2): {divisor}")

# --- Work-area pivot config: column order for Processed Data (15mins) D:L / N:V,
#     and each area's volume event type(s). Single source of truth. ---
PROC_AREAS = [
    {"report": "D.Analysis - OSR PiE",           "vol_events": {"MSKU", "PSKU"}},
    {"report": "D.Analysis - OSR Topup",         "vol_events": {"TPUT"}},
    {"report": "D.Analysis - E3 Packing",        "vol_events": {"PackingItemScannedEvent"}},
    {"report": "D.Analysis - Parcel Sortation",  "vol_events": {"ParcelSortedToSack"}},
    {"report": "D.Analysis - Parcel Induct",     "vol_events": {"SPAR"}},
    {"report": "D.Analysis - Inbound Decanting", "vol_events": {"DECN"}},
    {"report": "D.Analysis - OSR Decanting",     "vol_events": {"ODEC"}},
    {"report": "D.Analysis - BCR Inducting",     "vol_events": {"SPOS"}},
    {"report": "D.Analysis - E1/E2 Inducting",   "vol_events": {"SPOS"}},
]


In [0]:
try:
    ws = sh.worksheet("Data")
except gspread.exceptions.WorksheetNotFound:
    ws = sh.add_worksheet(title="Data", rows=2000, cols=len(COLUMNS))
    ws.update('A1', [COLUMNS])
    print("Created 'Data' tab with header")

In [0]:
# ============================================================
# WINDOW SELECTION - derived from BOTH the Data tab (which 15-min windows are
# already recorded) AND wall-clock (which windows are old enough to be safe
# to pull, i.e. BonusHub events have had time to land in the Delta table).
#
# The "newest safe" window is anchored to the current (possibly still-open)
# 15-min grid boundary - floor(now) - then stepped back one extra full grid
# step (SAFETY_MARGIN_MINUTES = 15), rather than computed from elapsed
# wall-clock time directly off `now`. This makes the result STABLE across an
# entire 15-min grid cell: a run scheduled at :01 that's delayed by Job
# Cluster cold-start and doesn't actually execute this cell until :11 (or
# anywhere up to :14) computes the exact SAME window as if it had run right
# on time at :01 - instead of drifting forward the later it happens to start.
#   e.g. floor(now) = 10:00 -> newest safe window = 09:30-09:45,
#        whether this cell actually runs at 10:01, 10:11, or 10:14.
# Minimum staleness is 15 min (cell runs right at the grid line); worst case
# (cell runs just under 15 min late) is just under 30 min. If a run is
# delayed PAST the next grid line, floor(now) simply advances and the window
# shifts forward with it - nothing is silently skipped, because any window a
# late run doesn't reach still shows up as a gap below and gets caught on a
# later run.
#
# Missing windows are found by diffing the full set of expected window-end
# times - walking back from the newest safe window, up to LOOKBACK_WINDOWS -
# against the "Date Time Range" values already written to Data. Using the
# full recorded set (not just a single max-watermark) means a hole anywhere
# in that horizon gets noticed and refilled, not just a late-starting run.
#
# Catch-up windows are processed NEWEST-FIRST (closest to now), each run
# working backwards through any backlog up to MAX_CATCHUP windows, so the
# live dashboard gets fresh data before older backlog is filled in.
#
# The floor/margin math is anchored to the job's SCHEDULED trigger time (via
# the trigger_time_iso parameter, see below), not to whenever this cell
# actually executes - so an arbitrarily long cold start no longer shifts
# which window gets selected. Without that parameter set, it falls back to
# real wall-clock time at execution, same behavior as before.
#
# Schedule the job with a minimal offset past each grid line (e.g. :01/:16/
# :31/:46) - it no longer needs tuning to outlast typical cold-start lag,
# since the margin above already absorbs it.
# ============================================================
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

uk = ZoneInfo("Europe/London")
utc = ZoneInfo("UTC")

dbutils.widgets.text("max_catchup_windows", "8", "Max Catch-up Windows")
MAX_CATCHUP = int(dbutils.widgets.get("max_catchup_windows"))

dbutils.widgets.text("lookback_windows", "96", "Lookback Windows (gap-scan horizon)")
LOOKBACK_WINDOWS = int(dbutils.widgets.get("lookback_windows"))

SAFETY_MARGIN_MINUTES = 15  # newest safe window ends this many minutes before
                             # the current 15-min grid boundary (floor(now)) -
                             # one full extra grid step back. See comment
                             # above for why this beats an elapsed-time offset.

def _parse_range_end_utc(dtr):
    try:
        end_str = dtr.split(' - ')[1].strip()
        end_local = datetime.strptime(end_str, '%d/%m/%Y %H:%M').replace(tzinfo=uk)
        return end_local.astimezone(utc)
    except Exception:
        return None

def _week_val(date_local):
    epoch_ref = datetime(2025, 12, 14).date()
    return ((date_local - epoch_ref).days // 7 + 46) % 52

# Anchor to the job's actual SCHEDULED trigger time, not whenever this cell
# happens to execute. A Job Cluster cold start can delay execution by 15+
# minutes past the nominal trigger, and datetime.now() only ever sees that
# later real clock - it has no way to know what time the job was *meant* to
# fire. Wire the job's trigger time in via a parameter using Databricks'
# dynamic value reference: set a job parameter named trigger_time_iso to
# {{job.trigger.time.iso_datetime}} in the job's Parameters config (Databricks
# resolves this to the scheduled fire time for schedule-triggered runs). If
# it's absent/unparseable (manual run, ad-hoc test, older job config), this
# falls back to real wall-clock time, same as before.
dbutils.widgets.text("trigger_time_iso", "", "Job Trigger Time (ISO, set via {{job.trigger.time.iso_datetime}})")
_trigger_raw = dbutils.widgets.get("trigger_time_iso").strip()
now_utc = datetime.now(utc)
if _trigger_raw:
    try:
        _anchor = datetime.fromisoformat(_trigger_raw.replace('Z', '+00:00'))
        anchor_utc = _anchor.astimezone(utc) if _anchor.tzinfo else _anchor.replace(tzinfo=utc)
    except Exception:
        anchor_utc = now_utc
else:
    anchor_utc = now_utc

floor_epoch = (int(anchor_utc.timestamp()) // 900) * 900
floor_utc = datetime.fromtimestamp(floor_epoch, utc)
newest_safe_end_utc = floor_utc - timedelta(minutes=SAFETY_MARGIN_MINUTES)

# Never reach back before today's local midnight (with a small grace period)
# - the Apps Script side (Archive.js) trims the live Data tab down to just
# today's date each run, so anything before that has already been archived
# off elsewhere and isn't something this pipeline should be re-adding to the
# live tab. PRE_MIDNIGHT_GRACE_MINUTES lets the scan still reach slightly
# before midnight, e.g. a run at 01:00 that finds 23:30-23:45 missing (from
# just before the Archive.js trim ran) can still catch and backfill it.
PRE_MIDNIGHT_GRACE_MINUTES = 30

today_local_date = anchor_utc.astimezone(uk).date()
start_of_today_utc = datetime(today_local_date.year, today_local_date.month, today_local_date.day, tzinfo=uk).astimezone(utc)
earliest_allowed_end_utc = start_of_today_utc - timedelta(minutes=PRE_MIDNIGHT_GRACE_MINUTES)

existing_rows = ws.get_all_values()[1:]  # skip header
covered_ends = set()
for _r in existing_rows:
    if len(_r) > 8 and _r[8].strip():
        _end = _parse_range_end_utc(_r[8].strip())
        if _end:
            covered_ends.add(_end)

missing_ends = []
cursor = newest_safe_end_utc
for _ in range(LOOKBACK_WINDOWS):
    if cursor <= earliest_allowed_end_utc:  # boundary itself is excluded, not just anything below it
        break
    if cursor not in covered_ends:
        missing_ends.append(cursor)
    cursor -= timedelta(minutes=15)

windows = []
for w_end_utc in missing_ends[:MAX_CATCHUP]:
    w_start_utc = w_end_utc - timedelta(minutes=15)
    w_start_local = w_start_utc.astimezone(uk)
    w_end_local = w_end_utc.astimezone(uk)
    windows.append({
        "start_utc": w_start_utc.strftime("%Y-%m-%d %H:%M:%S"),
        "end_utc":   w_end_utc.strftime("%Y-%m-%d %H:%M:%S"),
        "week_val":  _week_val(w_start_local.date()),
        "date_time_range": f"{w_start_local.strftime('%d/%m/%Y %H:%M')} - {w_end_local.strftime('%d/%m/%Y %H:%M')}",
    })

print(f"Actual execution time (UTC): {now_utc}")
print(f"Anchor used for window calc (UTC): {anchor_utc}" + ('  [from job trigger param]' if _trigger_raw else '  [wall-clock fallback - no trigger param set]'))
print(f"Current grid boundary floor(anchor) (UTC): {floor_utc}")
print(f"Newest safe window end (UTC): {newest_safe_end_utc}")
print(f"Start of today, local midnight (UTC): {start_of_today_utc}")
print(f"Earliest allowed window end, with grace (UTC): {earliest_allowed_end_utc}")
print(f"Missing windows in lookback horizon ({LOOKBACK_WINDOWS}): {len(missing_ends)}")
print(f"Windows to process this run (newest first): {len(windows)}")
for _w in windows:
    print(f"  {_w['date_time_range']}")
if len(missing_ends) > MAX_CATCHUP:
    print(f"WARNING: backlog exceeds max_catchup_windows ({MAX_CATCHUP}); "
          f"{len(missing_ends) - MAX_CATCHUP} older window(s) still pending — will catch up on subsequent runs.")
if not windows:
    print("No new windows to process.")


In [0]:
import pandas as pd
from datetime import datetime, timezone

# ============================================================
# CONSOLIDATED FETCH + BATCHED APPEND
# One Delta scan for the WHOLE run (all target windows x all reports), instead
# of len(windows) x len(reports) separate spark.sql().toPandas() calls, and ONE
# ws.append_rows() instead of one per (window, report).
#
# Every report reads the same table with the same warehouse filter and the same
# GROUP BY - they differ only by area code, event type and (for the two SPOS
# reports) a PAYLOAD_ATTRIBUTES predicate. So each raw event is tagged with the
# set of reports it belongs to (built as an array of CASE expressions, one per
# report, from the single-source `reports` config), then exploded. Using a
# *set* (array + explode) rather than a first-match CASE preserves the original
# semantics exactly: if a row satisfied two reports' filters it fed both, and it
# still does here - no reliance on the RET / PIE4EDW patterns being disjoint.
#
# The scan is bounded to [min start, max end] of this run's windows for
# partition pruning, then filtered to the EXACT 15-min buckets we intend to fill
# (window_epoch IN ...). That last filter matters when the gap-scan picks
# NON-contiguous windows (e.g. 09:00 and 11:00 missing but 10:00 already
# covered): a plain range scan would otherwise re-pull 10:00 and append
# duplicate rows for an already-covered window.
# ============================================================

failures = []
total_rows_pushed = 0

if not windows:
    print("No windows to process; skipping fetch/append.")
else:
    # --- Map each 15-min window bucket (UTC epoch, floored to the grid) to its
    #     metadata. Windows are already 15-min aligned, so the floor below and
    #     FLOOR(unix_timestamp/900)*900 in SQL land on the same epoch. ---
    def _win_epoch(s):
        return int(datetime.strptime(s, "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc).timestamp())

    win_by_epoch = {}
    for _w in windows:
        win_by_epoch[_win_epoch(_w['start_utc'])] = _w

    overall_start_utc = min(w['start_utc'] for w in windows)
    overall_end_utc = max(w['end_utc'] for w in windows)
    epoch_list = ", ".join(str(e) for e in sorted(win_by_epoch))

    # --- Turn one report's {area, event, extra} config into a standalone SQL
    #     boolean predicate (the `extra` field starts with "AND ", stripped). ---
    def _report_predicate(r):
        parts = []
        if r['area']:
            parts.append(f"PAYLOAD_AREACODE IN {r['area']}")
        parts.append(f"PAYLOAD_EVENTTYPE IN {r['event']}")
        extra = r['extra'].strip()
        if extra:
            if extra[:4].upper() == "AND ":
                extra = extra[4:].strip()
            parts.append(extra)
        return " AND ".join(f"({p})" for p in parts)

    preds = [(_report_predicate(r), r['name']) for r in reports]
    case_exprs = ",\n            ".join(f"CASE WHEN {pred} THEN '{name}' END" for pred, name in preds)
    match_any = " OR ".join(f"({pred})" for pred, _ in preds)

    query = f"""
      WITH base AS (
        SELECT
          date_format(from_utc_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP),'Europe/London'),'dd/MM/yyyy') AS Date,
          hour(from_utc_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP),'Europe/London')) AS Hour,
          CAST(FLOOR(unix_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP)) / 900) * 900 AS BIGINT) AS window_epoch,
          PAYLOAD_BONUSCODE AS bonus,
          PAYLOAD_EVENTTYPE AS eventtype,
          PAYLOAD_QUANTITY      AS qty,
          PAYLOAD_STANDARDHOURS AS std,
          PAYLOAD_SMV           AS smv,
          filter(array(
            {case_exprs}
          ), x -> x IS NOT NULL) AS report_names
        FROM landing_bonus_hub_event_parsed
        WHERE TRIM(PAYLOAD_WAREHOUSECODE) = 'X'
          AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) >= timestamp('{overall_start_utc}')
          AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) <  timestamp('{overall_end_utc}')
          AND ( {match_any} )
      )
      SELECT
        Date, Hour, window_epoch, bonus, eventtype, report_name,
        SUM(qty) AS Total_Quantity,
        SUM(std) AS Total_StandardHours,
        SUM(smv) AS Total_SMV
      FROM base
      LATERAL VIEW explode(report_names) t AS report_name
      WHERE window_epoch IN ({epoch_list})
      GROUP BY Date, Hour, window_epoch, bonus, eventtype, report_name
    """

    report_order = {r['name']: i for i, r in enumerate(reports)}

    # --- Single scan. A query-level failure is all-or-nothing now (same
    #     templated SQL used to fail every report anyway); it's captured so the
    #     Failures tab still records it and cell 9 still raises. ---
    pdf = None
    try:
        pdf = spark.sql(query).toPandas()
        print(f"Consolidated query returned {len(pdf)} grouped rows "
              f"across {len(windows)} window(s) x {len(reports)} report(s).")
    except Exception as e:
        error_msg = f"{type(e).__name__}: {str(e)}"
        print(f"CONSOLIDATED QUERY FAILED - {error_msg}")
        failures.append({"report": "(consolidated query)", "error": error_msg,
                         "window": f"{overall_start_utc}..{overall_end_utc} UTC"})

    if pdf is not None:
        # --- Shape into the Data-tab COLUMNS order; note which windows had data. ---
        rows = []
        windows_with_data = set()
        for _, row in pdf.iterrows():
            ep = int(row['window_epoch'])
            w = win_by_epoch.get(ep)
            if w is None:
                continue  # safety: outside the target buckets
            windows_with_data.add(ep)
            hour_key = int(row['Hour']) if str(row['Hour']).strip().lstrip('-').isdigit() else 0
            rows.append({
                "sort": (ep, report_order.get(row['report_name'], 999), hour_key, str(row['bonus'])),
                "vals": [
                    row['Date'], row['Hour'], row['bonus'], row['eventtype'],
                    row['Total_Quantity'], row['Total_StandardHours'], row['Total_SMV'],
                    w['week_val'], w['date_time_range'], row['report_name'],
                ],
            })

        # --- One placeholder per EMPTY window so the gap-scan (cell 6) marks it
        #     covered and doesn't re-fetch it forever. cell 8 skips these (Hour
        #     == ''). One per window is enough (was one per report per window). ---
        for ep, w in win_by_epoch.items():
            if ep not in windows_with_data:
                rows.append({
                    "sort": (ep, 999, 0, ''),
                    "vals": ['', '', '', '(no data this window)', '', '', '',
                             w['week_val'], w['date_time_range'], '(no data this window)'],
                })

        rows.sort(key=lambda d: d['sort'])
        values = [[str(c) for c in d['vals']] for d in rows]

        # --- ONE batched append (was up to len(windows) x len(reports) writes,
        #     which also brushed the Sheets 60-writes/min quota). ---
        if values:
            try:
                ws.append_rows(values, value_input_option='USER_ENTERED', table_range='A1')
                total_rows_pushed = len(values)
                print(f"Appended {total_rows_pushed} rows in one batch.")
            except Exception as e:
                error_msg = f"{type(e).__name__}: {str(e)}"
                print(f"BATCH APPEND FAILED - {error_msg}")
                failures.append({"report": "(sheet append)", "error": error_msg,
                                 "window": f"{len(windows)} window(s)"})

        # --- Per-window x report breakdown for the run log. ---
        counts = pdf.groupby(['window_epoch', 'report_name']).size()
        for ep in sorted(win_by_epoch):
            w = win_by_epoch[ep]
            print(f"\n=== Window: {w['date_time_range']} ===")
            any_rows = False
            for r in reports:
                n = int(counts.get((ep, r['name']), 0))
                if n:
                    any_rows = True
                    print(f"  [{r['name']}] {n} rows")
            if not any_rows:
                print("  (no data this window)")

print(f"\n--- Run summary ---")
print(f"Windows processed: {len(windows)}")
print(f"Total rows pushed: {total_rows_pushed}")
print(f"Reports failed: {len(failures)} / {len(windows) * len(reports)}")


In [0]:
# ============================================================
# REBUILD "Processed Data (15mins)" from the Data tab
# Runs AFTER this window's rows are appended above, so the rebuild always
# includes them. Pivot: group raw Data rows by (Date Time Range, Hour, Bonus).
#   D:L = sum(StandardHours) per area / divisor ; M = total ;
#   N:V = sum(Quantity) per area for that area's volume event(s).
# Databricks is the sole writer of this tab (Apps Script no longer processes).
# ============================================================
from datetime import datetime
from collections import OrderedDict

def _f(x):
    try:
        return float(str(x).replace(',', '').strip())
    except Exception:
        return 0.0

def _parse_start(dtr):
    try:
        return datetime.strptime(dtr.split(' - ')[0].strip(), '%d/%m/%Y %H:%M')
    except Exception:
        return datetime.max

area_index = {a["report"]: i for i, a in enumerate(PROC_AREAS)}
n_areas = len(PROC_AREAS)

all_data = ws.get_all_values()            # ws = Data worksheet (from earlier cell)
data_rows = all_data[1:] if all_data else []

# Data cols: A=Date B=Hour C=Bonus D=EventType E=Qty F=StdHours G=SMV H=Week I=DateTimeRange J=ReportName
groups = OrderedDict()
for r in data_rows:
    if len(r) < 10:
        continue
    hour = r[1].strip()
    if hour == '':                        # skip "(no data this window)" placeholders
        continue
    ai = area_index.get(r[9].strip())     # Report Name
    if ai is None:
        continue
    key = (r[8].strip(), hour, r[2].strip())   # (Date Time Range, Hour, Bonus)
    g = groups.get(key)
    if g is None:
        g = {"std": [0.0] * n_areas, "vol": [0.0] * n_areas}
        groups[key] = g
    g["std"][ai] += _f(r[5])              # Total_StandardHours (all event types)
    if r[3].strip() in PROC_AREAS[ai]["vol_events"]:
        g["vol"][ai] += _f(r[4])          # Total_Quantity (volume event only)

out = []
for (dtr, hour, bonus), g in groups.items():
    std = [s / divisor for s in g["std"]]
    row = [dtr, hour, bonus] + std + [sum(std)] + g["vol"]
    out.append((_parse_start(dtr), bonus, row))

out.sort(key=lambda t: (t[0], t[1]))
proc_values = [t[2] for t in out]

proc_ws = sh.worksheet("Processed Data (15mins)")
proc_ws.batch_clear(["A2:V"])
if proc_values:
    proc_ws.update("A2", proc_values, value_input_option="USER_ENTERED")

print(f"Processed Data (15mins) rebuilt: {len(proc_values)} rows")

In [0]:
if failures:
    try:
        fail_ws = sh.worksheet("Failures")
        fail_existing = fail_ws.col_values(1)
        fail_next_row = len(fail_existing) + 1
    except gspread.exceptions.WorksheetNotFound:
        fail_ws = sh.add_worksheet(title="Failures", rows=1000, cols=4)
        fail_ws.update('A1', [['Window', 'Report', 'Error']])
        fail_next_row = 2

    fail_rows = [[f['window'], f['report'], f['error']] for f in failures]
    fail_ws.update(f'A{fail_next_row}', fail_rows, value_input_option='USER_ENTERED')

    raise Exception(f"{len(failures)} report(s) failed this run: {[f['report'] for f in failures]}")